# MediCare Patient Follow-Up Agent

## Setup & Imports

In [3]:
# Install dependencies
!pip install langchain langchain-openai pandas tabulate -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import pandas as pd
import json
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="openrouter/owl-alpha",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
)
print("✅ LLM initialised:", llm.model)

✅ LLM initialised: openrouter/owl-alpha


## Task 1 — Data Loading & Exploratory Analysis

In [3]:
df = pd.read_csv("patient_data.csv", parse_dates=["last_visit_date", "next_scheduled_visit"])

print(f"Dataset: {df.shape[0]} patients × {df.shape[1]} columns")
df.head(3)

Dataset: 100 patients × 18 columns


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,2024-10-03,149,No,132,86,64,91,Medication dose adjusted this visit.


In [4]:
# Quick EDA 

print("=" * 55)
print("TASK 1 — EXPLORATORY DATA ANALYSIS")
print("=" * 55)

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDiagnosis Distribution:\n{df['diagnosis'].value_counts().head(10)}")
print(f"\nMissed Appointments:\n{df['missed_last_appointment'].value_counts().to_dict()}")
print(f"\nAvg Days Since Last Visit: {df['days_since_last_visit'].mean():.1f}")
print(f"\nOverdue > 180 days: {(df['days_since_last_visit'] > 180).sum()} patients")
print(f"\nLab Test Types:\n{df['lab_test'].value_counts()}")
print(f"\nAge Stats:\n{df['age'].describe()}")
print(f"\nNull Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

TASK 1 — EXPLORATORY DATA ANALYSIS

Shape: (100, 18)

Columns: ['patient_id', 'patient_name', 'age', 'gender', 'diagnosis', 'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']

Diagnosis Distribution:
diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
COPD                              7
Osteoarthritis                    7
Type 2 Diabetes, Hypertension     7
Depression                        7
Type 2 Diabetes                   6
COPD, Heart Failure               6
Asthma                            6
Name: count, dtype: int64

Missed Appointments:
{'No': 85, 'Yes': 15}

Avg Days Since Last Visit: 211.4

Overdue > 180 days: 57 patients

Lab Test Types:
lab_test
BP Systolic    18
HbA1c          17
FEV1%          13
BNP            12
TS

In [5]:
print("=== Diagnosis Distribution ===")
print(df["diagnosis"].value_counts().head(10).to_string())

print("\n=== Missed Appointments ===")
print(df["missed_last_appointment"].value_counts().to_string())

print("\n=== Key Vitals Summary ===")
print(df[["vitals_bp_systolic", "vitals_bp_diastolic", "vitals_heart_rate", "vitals_spo2"]].describe().round(1).to_string())

print("\n=== Days Since Last Visit ===")
print(df["days_since_last_visit"].describe().round(1).to_string())

=== Diagnosis Distribution ===
diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
COPD                              7
Osteoarthritis                    7
Type 2 Diabetes, Hypertension     7
Depression                        7
Type 2 Diabetes                   6
COPD, Heart Failure               6
Asthma                            6

=== Missed Appointments ===
missed_last_appointment
No     85
Yes    15

=== Key Vitals Summary ===
       vitals_bp_systolic  vitals_bp_diastolic  vitals_heart_rate  vitals_spo2
count               100.0                100.0              100.0        100.0
mean                145.6                 87.7               82.4         94.5
std                  20.4                 13.6               14.5          2.8
min                 105.0                 65.0               58.0         90.0
25%                 130.0                 75.8               72.0         92.0
50%                 145.

## Task 2 — Define Agent Tools

In [6]:
# Clinical thresholds reference
THRESHOLDS = {
    "HbA1c":      {"high": 7.0,   "unit": "%",    "condition": "Type 2 Diabetes"},
    "BNP":        {"high": 100,   "unit": "pg/mL","condition": "Heart Failure"},
    "Creatinine": {"high": 1.2,   "unit": "mg/dL","condition": "CKD"},
    "Hemoglobin": {"low":  12.0,  "unit": "g/dL", "condition": "Anemia"},
    "FEV1":       {"low":  70.0,  "unit": "%",    "condition": "COPD"},
}

@tool
def get_patient_record(patient_id: str) -> str:
    """Retrieve full clinical record for a single patient by patient_id (e.g. 'P0001')."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return f"No patient found with ID {patient_id}."
    return row.to_dict(orient="records")[0].__str__()


@tool
def flag_lab_risk(patient_id: str) -> str:
    """Check whether a patient's latest lab value exceeds clinical thresholds. Returns risk level and explanation."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    lab, val = r["lab_test"], float(r["lab_value"])
    thresh = THRESHOLDS.get(lab)
    if not thresh:
        return f"No threshold defined for lab test '{lab}'."
    if "high" in thresh and val > thresh["high"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold > {thresh['high']}). Condition: {thresh['condition']}.")
    if "low" in thresh and val < thresh["low"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold < {thresh['low']}). Condition: {thresh['condition']}.")
    return f"NORMAL — {lab} = {val} {thresh['unit']} within acceptable range."


@tool
def flag_vitals_risk(patient_id: str) -> str:
    """Assess whether a patient's blood pressure, heart rate, or SpO2 are outside safe ranges."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    flags = []
    if r["vitals_bp_systolic"] >= 140 or r["vitals_bp_diastolic"] >= 90:
        flags.append(f"Hypertension: BP {r['vitals_bp_systolic']}/{r['vitals_bp_diastolic']} mmHg")
    if r["vitals_heart_rate"] > 100:
        flags.append(f"Tachycardia: HR {r['vitals_heart_rate']} bpm")
    if r["vitals_spo2"] < 92:
        flags.append(f"Hypoxia: SpO2 {r['vitals_spo2']}%")
    return "VITALS RISKS: " + "; ".join(flags) if flags else "Vitals within normal limits."


@tool
def get_missed_appointment_patients(top_n: int = 10) -> str:
    """Return patients who missed their last appointment, sorted by days since last visit (most overdue first)."""
    missed = (df[df["missed_last_appointment"].str.lower() == "yes"]
              .sort_values("days_since_last_visit", ascending=False)
              .head(top_n)[["patient_id", "patient_name", "diagnosis",
                            "days_since_last_visit", "next_scheduled_visit"]])
    if missed.empty:
        return "No missed appointments found."
    return missed.to_string(index=False)


@tool
def generate_care_action_plan(patient_id: str, risk_summary: str) -> str:
    """Given a patient_id and a plain-text risk_summary, produce a structured follow-up care action plan."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    plan = {
        "patient_id": patient_id,
        "patient_name": r["patient_name"],
        "diagnosis": r["diagnosis"],
        "risk_summary": risk_summary,
        "recommended_actions": "[LLM will fill this in]",
        "priority": "[LLM will fill this in]",
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    return json.dumps(plan, indent=2)


tools = [get_patient_record, flag_lab_risk, flag_vitals_risk,
         get_missed_appointment_patients, generate_care_action_plan]
print(f"✅ {len(tools)} tools registered:", [t.name for t in tools])

✅ 5 tools registered: ['get_patient_record', 'flag_lab_risk', 'flag_vitals_risk', 'get_missed_appointment_patients', 'generate_care_action_plan']


## Task 3 — Implement the Agentic Loop

In [9]:
SYSTEM_PROMPT = """You are MediCare's AI clinical follow-up agent.
Your job is to analyse exactly one patient at a time.

When the user asks for a patient, you must:
1. Use the requested patient_id only.
2. Retrieve that patient's record and assess their clinical risks.
3. Generate a focused, prioritised follow-up care plan for that one patient.

Important constraints:
- Never analyse, list, or summarise other patients.
- Never produce a population report, ranked list of multiple patients, or batch summary.
- Do not mention other patient_ids unless the user explicitly asks.
- Do not use tools that return multiple patients unless the user explicitly requests a population view.
- Be concise, clinical, and prioritise patient safety.
- Always ground your decisions in tool outputs — do not guess or hallucinate values."""

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

/tmp/ipykernel_4286/2287331848.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


## Task 4 — Single Patient Analysis

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage

PATIENT_ID = "P0001"   # Change to any patient_id from the CSV

result = agent_executor.invoke({
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Analyse only patient {PATIENT_ID}. "
            f"Use the provided patient_id directly, do not ask for more information, "
            f"and generate a specific prioritised follow-up action plan for this one patient only."
        )),
    ]
})
print("\n" + "="*60)
print("AGENT OUTPUT")
print("="*60)
print(result["messages"][-1].content)


AGENT OUTPUT
## Patient Analysis: P0001 — Aarav Sharma (68M)

### Risk Assessment
| Domain | Finding | Status |
|---|---|---|
| **HbA1c** | **9.9%** (target <7.0%) | 🔴 HIGH RISK |
| **Blood Pressure** | **174/70 mmHg** (systolic hypertension) | 🔴 HIGH RISK |
| **Heart Rate** | 95 bpm | 🟡 Borderline high |
| **SpO2** | 96% | 🟢 Normal |
| **Last Visit** | 251 days ago (April 2024) | 🟡 Gap in follow-up |

### Prioritized Follow-Up Action Plan

**Priority: URGENT — Schedule appointment within 7 days**

1.🏴️ **Glycemic Control (Immediate)**
   - HbA1c of 9.9% indicates poorly controlled diabetes despite good adherence to Metformin 500mg.
   - **Action:** Up-titrate Metformin (e.g., increase to 1000mg BID) OR add a second-line agent (e.g., GLP-1 RA or SGLT2 inhibitor). Consider insulin evaluation if progression continues.
   - **Target:** HbA1c <7.0% within 3 months.

2.️ **Blood Pressure Management (Immediate)**
   - Systolic BP of 174 mmHg is significantly elevated (Hypertension Stage 2),

## Task 5 — Missed Appointment Follow-Up

In [ ]:
# Step 1: Identify top missed-appointment patients
missed_result = agent_executor.invoke({
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=(
            "Identify the top 5 patients who missed their last appointment. "
            "Use the missed-appointment tool and return the patients sorted by days overdue, "
            "showing patient_id, patient_name, diagnosis, and days_since_last_visit."
        )),
    ]
})
print(missed_result["messages"][-1].content)

Here are the **top 5 patients who missed their last appointment**, sorted by days overdue:

| Patient ID | Patient Name | Diagnosis | Days Since Last Visit | Next Scheduled Visit |
|------------|-------------|-----------|----------------------|---------------------|
| P0099 | Paramita Chatterjee | Type 2 Diabetes, Hypertension | 340 | 2024-03-26 |
| P0066 | Angela Pereira | Type 2 Diabetes, CKD | 322 | 2024-03-29 |
| P0043 | Gaurav Dey | Hypertension, Anemia | 311 | 2024-05-24 |
| P0036 | Vandana Srivastava | Type 2 Diabetes | 274 | 2024-05-31 |
| P0062 | Mary Joseph | Asthma | 271 | 2024-05-19 |

**Key observations:**
- All 5 patients are significantly overdue (271–340 days), indicating a serious gap in care continuity.
- The top 2 patients (P0099, P0066) have **Type 2 Diabetes with comorbidities** (Hypertension and CKD, respectively) — these are high-risk conditions that require regular monitoring.
- P0043 has Hypertension and Anemia, also warranting prompt follow-up.

If you'd like 

In [ ]:
# Step 2: Batch analysis — run the agent for each missed-appointment patient
missed_patients = (
    df[df["missed_last_appointment"].str.lower() == "yes"]
    .sort_values("days_since_last_visit", ascending=False)
    .head(5)["patient_id"]
    .tolist()
)
print(f"Analysing {len(missed_patients)} overdue patients: {missed_patients}\n")

batch_results = []
for pid in missed_patients:
    print(f"--- Analysing {pid} ---")
    r = agent_executor.invoke({
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=(
                f"Analyse only patient {pid}. "
                f"Assess their clinical risks and generate a follow-up care action plan for this one patient only."
            )),
        ]
    })
    batch_results.append({"patient_id": pid, "plan": r["messages"][-1].content})
    print(r["messages"][-1].content)
    print()

Analysing 5 overdue patients: ['P0099', 'P0066', 'P0043', 'P0036', 'P0062']

--- Analysing P0099 ---
## Clinical Analysis: Patient P0099 — Paramita Chatterjee

### Patient Overview
- **Age/Gender:** 70-year-old Male
- **Diagnoses:** Type 2 Diabetes, Hypertension
- **Current Medications:** Metformin 500mg, Amlodipine 5mg
- **Last Visit:** January 26, 2024 (340 days ago)
- **Missed Last Appointment:** Yes

---

### Risk Assessment

| Parameter | Value | Status |
|-----------|-------|--------|
| **HbA1c** | 9.8% | 🔴 HIGH RISK (target <7.0%) |
| **Blood Pressure** | 121/101 mmHg | 🔴 Diastolic hypertension |
| **SpO2** | 91% | 🔴 Hypoxia |
| **Heart Rate** | 71 bpm | 🟢 Normal |

---

### Prioritized Follow-Up Care Action Plan

**🔴 PRIORITY 1 — URGENT (Within 24–48 hours)**

1. **Hypoxia Evaluation (SpO2 91%)**
   - Immediate pulse oximetry recheck and respiratory assessment
   - Order chest X-ray and consider arterial blood gas (ABG)
   - Evaluate for underlying causes: pneumonia, COPD exace

In [17]:
# Step 3: Summary dashboard of all batch results
print("="*60)
print("MISSED APPOINTMENT FOLLOW-UP — SUMMARY DASHBOARD")
print("="*60)
for item in batch_results:
    print(f"\n[{item['patient_id']}]")
    # Extract first 3 lines of each plan for the summary view
    print("\n".join(item["plan"].split("\n")[:6]))
    print("...")

MISSED APPOINTMENT FOLLOW-UP — SUMMARY DASHBOARD

[P0099]
## Clinical Analysis: Patient P0099 — Paramita Chatterjee

### Patient Overview
- **Age/Gender:** 70-year-old Male
- **Diagnoses:** Type 2 Diabetes, Hypertension
- **Current Medications:** Metformin 500mg, Amlodipine 5mg
...

[P0066]
## Patient P0066 — Angela Pereira (37, Male)

### Clinical Risk Assessment

| Parameter | Value | Risk Level |
|---|---|---|
...

[P0043]
## Clinical Analysis: Patient P0043 - Gaurav Dey

### Patient Overview
- **Age/Gender:** 69-year-old Male
- **Diagnoses:** Hypertension, Anemia
- **Current Medications:** Amlodipine 5mg, Ferrous Sulfate 200mg
...

[P0036]
## Patient P0036 — Clinical Risk Assessment & Follow-Up Care Plan

**Patient:** Vandana Srivastava, 71F
**Diagnosis:** Type 2 Diabetes

---
...

[P0062]
## Clinical Analysis & Follow-Up Care Plan for Patient P0062

### Patient Summary
- **Name:** Mary Joseph | **Age/Gender:** 35M
- **Diagnosis:** Asthma
- **Current Medication:** Salbutamol 100mcg